# Sentiment Analysis

Sentiment analysis on guest feedback and reviews.

In [1]:
from transformers import pipeline
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import torch

In [2]:
# mount drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [65]:
# processed data loading
df = pd.read_csv("drive/MyDrive/processed/reviews_processed.csv")
print("Shape:", df.shape )
df.head()

Shape: (5946, 6)


,review_text,language,length,is_long_review,normalized_text,final_text
0,Chambre confortable mais décoration un peu dém...,fr,15,False,chambre confortable mais decoration un peu dem...,chambre confortable decoration peu demodee. pe...
1,Le spa propose un traitement signature exclusi...,fr,20,False,le spa propose un traitement signature exclusi...,spa propose traitement signature exclusif deve...
2,Un séjour correct mais qui ne justifie pas ple...,fr,23,False,un sejour correct mais qui ne justifie pas ple...,sejour correct qui ne justifie pas_pleinement ...
3,The laundry service express saved our gala din...,en,12,False,the laundry service express saved our gala din...,laundry service express saved gala dinner foll...
4,L'exposition de collection d'art contemporain ...,fr,24,False,l'exposition de collection d'art contemporain ...,l'exposition collection d'art contemporain l'h...


In [2]:
# processed data loading
df = pd.read_csv("../data/processed/reviews_processed.csv")
print("Shape:", df.shape )
df.head()

Shape: (5946, 6)


,review_text,language,length,is_long_review,normalized_text,final_text
0,Chambre confortable mais décoration un peu dém...,fr,15,False,chambre confortable mais decoration un peu dem...,chambre confortable decoration peu demodee. pe...
1,Le spa propose un traitement signature exclusi...,fr,20,False,le spa propose un traitement signature exclusi...,spa propose traitement signature exclusif deve...
2,Un séjour correct mais qui ne justifie pas ple...,fr,23,False,un sejour correct mais qui ne justifie pas ple...,sejour correct qui ne justifie pas_pleinement ...
3,The laundry service express saved our gala din...,en,12,False,the laundry service express saved our gala din...,laundry service express saved gala dinner foll...
4,L'exposition de collection d'art contemporain ...,fr,24,False,l'exposition de collection d'art contemporain ...,l'exposition collection d'art contemporain l'h...


In [32]:
df["normalized_text"].iloc[1000]

'what more can i say about hotel de la promenade than its beautiful, super clean, friendly, welcoming i could go on and on and on... wow! i visited ottawa with my boyfriend and friends in the first week of september and it was superb, the hotel was undergoing some renovation but never once seen a builder or a dust sheet nor did we hear a drill or rubble or any noise from these works. hotel de la promenade has the most beautiful ballroom what could make you imagine you could be part of phantom of the opera it was captivating, along with the elegant hallways and twinkling chandeliers, and all the staff are so friendly which are also a credit to the wonderful hotel, the ladys in house keeping were also so friendly and chirpy in the morning, so much so you looked forward to meeting them in the hallway in the morning. we stayed for 1 week and the only fault we could find and thats hard to fault was the price of the breakfast, but wow what a breakfast. i for one said that the stay at hotel d

**Vader**

In [6]:
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [71]:
def vader_score(text: str) -> float:
    return sia.polarity_scores(str(text))["compound"]

df["vader_score"]  = df["normalized_text"].apply(vader_score)
df["vader_score"].describe()

,vader_score
count,5946.000000
mean,0.642451
std,0.511924
min,-0.994000
25%,0.440400
50%,0.928400
75%,0.980200
max,1.000000


In [72]:
# score to label
def vader_label(score: float) -> str:
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"
    
df["vader_label"] = df["vader_score"].apply(vader_label)
df["vader_label"].value_counts(normalize=True)

,proportion
vader_label,
positive,0.802220
neutral,0.102926
negative,0.094854


In [9]:
df.sample(10, random_state=42)[["normalized_text", "vader_score", "vader_label"]]

,normalized_text,vader_score,vader_label
2264,i really had high expectations of good service...,0.9300,positive
3669,we were excited to stay at hotel de la promena...,0.9533,positive
1406,stayed at hotel de la promenade a few days aft...,0.9966,positive
5858,personnel disponible pour des services: transp...,0.0000,neutral
3666,i was a little hesitant to book this hotel aft...,0.9264,positive
2526,we went there with my parents for their weddin...,0.9670,positive
3711,i stayed at hotel de la promenade two weeks ag...,-0.9530,negative
4015,this hotel has a fantastic location which is v...,0.8934,positive
1726,several of my colleagues and i stayed here for...,0.9967,positive
5030,great location to experience ottawa. the room ...,0.9625,positive


In [10]:
df.sort_values("vader_score").head(5)[["normalized_text","vader_score","vader_label"]]
df.sort_values("vader_score").tail(5)[["normalized_text","vader_score","vader_label"]]  

,normalized_text,vader_score,vader_label
1531,my girlfriends and i had a wonderful 2 night s...,0.9993,positive
870,from the moment the doorman assisted us with o...,0.9993,positive
4712,having seen some of the trip advisor reviews w...,0.9995,positive
4490,"first, my apologies - this review is long over...",0.9997,positive
1665,i had the opportunity to stay at hotel de la p...,1.0000,positive


In [ ]:
# extremes review vader score make sense 

**Transformer**

In [28]:
from transformers import pipeline

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
device = 0 if torch.cuda.is_available() else -1
sent_pipe = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    truncation=True,
    max_length=256,
    device=device
)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [42]:
df_small = df.sample(500, random_state=42).copy()

res = sent_pipe(df_small["normalized_text"].tolist(), batch_size=8)

df_small["tr_label"] = [r["label"] for r in res]
df_small["tr_score"] = [r["score"] for r in res]

df_small["tr_label"].value_counts(normalize=True)

,proportion
tr_label,
4 stars,0.378
5 stars,0.298
3 stars,0.142
2 stars,0.130
1 star,0.052


In [66]:
res_full = sent_pipe(df["normalized_text"].tolist(), batch_size=32)

df["tr_label"] = [r["label"] for r in res_full]
df["tr_score"] = [r["score"] for r in res_full]

df["tr_label"].value_counts(normalize=True)

,proportion
tr_label,
4 stars,0.385806
5 stars,0.297511
2 stars,0.137403
3 stars,0.134040
1 star,0.045240


In [67]:
def map_stars(label):
    stars = int(label[0])  # first character
    if stars >= 4:
        return "positive"
    elif stars == 3:
        return "neutral"
    else:
        return "negative"

df["tr_label"] = df["tr_label"].apply(map_stars)

df["tr_label"].value_counts(normalize=True)

,proportion
tr_label,
positive,0.683317
negative,0.182644
neutral,0.134040


In [73]:
agreement = (df["vader_label"] == df["tr_label"]).mean()
agreement

np.float64(0.6907164480322906)

In [34]:
disagree = df[df["vader_label"] != df["tr_label"]]
disagree.sample(10)[["normalized_text","vader_label","tr_label"]]

,normalized_text,vader_label,tr_label
1423,stayed there because we thought we got a great...,positive,negative
3139,stayed here for a total of four nights and it ...,positive,negative
3314,for a first time visitor and to that fact an a...,positive,neutral
5538,my husband and i have just returned from our s...,positive,negative
792,la piscine interieure est magnifiquement concu...,neutral,positive
3411,hotel de la promenade was okay - we had a smal...,positive,neutral
2950,"as a hilton honors diamond member, we have spe...",positive,negative
344,le service de voiturier est efficace et courto...,neutral,positive
1982,hotel de la promenade is well-situated but so ...,negative,neutral
337,restaurant proposant une cuisine de qualite ma...,positive,neutral


In [38]:
df["normalized_text"].iloc[2950	]

"as a hilton honors diamond member, we have spent years staying loyal to the brand, spending tens of thousands of dollars on stays. we are used to upgrades, and being treated really well as loyal guests of the hilton chain. this came to a halting stop at hotel de la promenade. yes, the lobby is grand, beautiful, festive, historic..etc. the staff, however, is incredibly rude, nonchalant, and cannot be bothered by the lowly like of those who would have the audacity to pay a *gasp* advanced purchase rate. youd think a hotel would appreciate the fact that a guest would pay 100% up front, ensuring they will have that room sold, but instead, at check-in we were met with this exact phrase, i see you are a diamond member, but since you paid the advanced purchase rate, you will have to use starbucks coupons for breakfast. now, i know that a starbucks coupon is appealing to 99% of the population, but when you are used to full hot breakfast, and they are parading people past the most extravagant 

In [74]:
df.groupby("language")["tr_label"].value_counts(normalize=True)

language  tr_label
en        positive    0.701688
          negative    0.178749
          neutral     0.119563
fr        positive    0.581319
          neutral     0.214286
          negative    0.204396
other     positive    1.000000
Name: proportion, dtype: float64

In [40]:
df.groupby("is_long_review")["tr_label"].value_counts(normalize=True)

is_long_review  tr_label
False           positive    0.687241
                negative    0.180345
                neutral     0.132414
True            positive    0.527397
                negative    0.273973
                neutral     0.198630
Name: proportion, dtype: float64

In [50]:
df

,review_text,language,length,is_long_review,normalized_text,final_text,vader_score,vader_label,tr_label,tr_score
0,Chambre confortable mais décoration un peu dém...,fr,15,False,chambre confortable mais decoration un peu dem...,chambre confortable decoration peu demodee. pe...,0.0000,neutral,neutral,0.748015
1,Le spa propose un traitement signature exclusi...,fr,20,False,le spa propose un traitement signature exclusi...,spa propose traitement signature exclusif deve...,0.0000,neutral,positive,0.447036
2,Un séjour correct mais qui ne justifie pas ple...,fr,23,False,un sejour correct mais qui ne justifie pas ple...,sejour correct qui ne justifie pas_pleinement ...,0.0000,neutral,neutral,0.692643
3,The laundry service express saved our gala din...,en,12,False,the laundry service express saved our gala din...,laundry service express saved gala dinner foll...,0.4215,positive,positive,0.619511
4,L'exposition de collection d'art contemporain ...,fr,24,False,l'exposition de collection d'art contemporain ...,l'exposition collection d'art contemporain l'h...,0.0000,neutral,positive,0.462993
...,...,...,...,...,...,...,...,...,...,...
5941,Très bien situé (station de Vanier et autobus ...,fr,65,False,tres bien situe (station de vanier et autobus ...,"bien situe (station vanier autobus tout pres, ...",-0.1280,negative,positive,0.591498
5942,Très bon hôtel et très très bien situé. Les ch...,fr,64,False,tres bon hotel et tres tres bien situe. les ch...,bon hotel bien situe. chambres sont petites pr...,-0.3612,negative,positive,0.814970
5943,Le plus important quand on est dans une ville ...,fr,43,False,le plus important quand on est dans une ville ...,"important quand ville comme ottawa, nest-il pa...",0.7506,positive,positive,0.551225
5944,Hotel De La Promenade est très bien situé et l...,fr,53,False,hotel de la promenade est tres bien situe et l...,hotel promenade bien situe renovation chambres...,0.0000,neutral,positive,0.558370


In [75]:
df.rename(columns={
    "tr_label": "sentiment_label",
    "tr_score": "sentiment_score"
}, inplace=True)

In [76]:
df

,review_text,language,length,is_long_review,normalized_text,final_text,sentiment_label,sentiment_score,vader_score,vader_label
0,Chambre confortable mais décoration un peu dém...,fr,15,False,chambre confortable mais decoration un peu dem...,chambre confortable decoration peu demodee. pe...,neutral,0.748015,0.0000,neutral
1,Le spa propose un traitement signature exclusi...,fr,20,False,le spa propose un traitement signature exclusi...,spa propose traitement signature exclusif deve...,positive,0.447036,0.0000,neutral
2,Un séjour correct mais qui ne justifie pas ple...,fr,23,False,un sejour correct mais qui ne justifie pas ple...,sejour correct qui ne justifie pas_pleinement ...,neutral,0.692643,0.0000,neutral
3,The laundry service express saved our gala din...,en,12,False,the laundry service express saved our gala din...,laundry service express saved gala dinner foll...,positive,0.619511,0.4215,positive
4,L'exposition de collection d'art contemporain ...,fr,24,False,l'exposition de collection d'art contemporain ...,l'exposition collection d'art contemporain l'h...,positive,0.462993,0.0000,neutral
...,...,...,...,...,...,...,...,...,...,...
5941,Très bien situé (station de Vanier et autobus ...,fr,65,False,tres bien situe (station de vanier et autobus ...,"bien situe (station vanier autobus tout pres, ...",positive,0.591498,-0.1280,negative
5942,Très bon hôtel et très très bien situé. Les ch...,fr,64,False,tres bon hotel et tres tres bien situe. les ch...,bon hotel bien situe. chambres sont petites pr...,positive,0.814970,-0.3612,negative
5943,Le plus important quand on est dans une ville ...,fr,43,False,le plus important quand on est dans une ville ...,"important quand ville comme ottawa, nest-il pa...",positive,0.551225,0.7506,positive
5944,Hotel De La Promenade est très bien situé et l...,fr,53,False,hotel de la promenade est tres bien situe et l...,hotel promenade bien situe renovation chambres...,positive,0.558370,0.0000,neutral


In [79]:
df.to_csv("drive/MyDrive/processed/reviews_with_sentiment.csv", index = False)

**Sentiment Analysis: VADER vs Transformer Model Comparison**

This analysis compared two sentiment analysis approaches on 5,946 hotel reviews in English and French.

**Methodology**

- VADER: Lexicon-based sentiment analyzer, fast but context-limited
- Transformer: BERT-based multilingual model, contextually aware

**VADER Results**

- Sentiment distribution: 10 % neutral, 80 % positive, 9 % negative
- Scores range from -1 to 1
- Quick execution but less nuanced understanding

**Transformer Model Results**

- Sentiment distribution: 68 % positive, 13 % neutral, 18 % negative
- Deeper contextual understanding of reviews
- Stricter evaluation, capturing nuances missed by lexicon methods

**Model Comparison**

- Agreement rate: 69.1%
- 1,839 cases of disagreement (30.9%)
- Primary disagreement: VADER predicts neutral while Transformer predicts positive, especially in French

**Language & Length Effects**

- Transformer is 12% less positive on French reviews compared to English
- Longer reviews receive 16 % more negative scores from Transformer model
- Both models show language-dependent behavior

**Key Findings**

1. Transformer model better captures contextual nuance and sarcasm
2. VADER's neutral category acts as catch-all for mixed sentiments
3. Language and review length significantly impact predictions
4. Methods complement each other for comprehensive sentiment analysis
5. The majority of reviews are positive, confirming strong overall guest satisfaction. 
6. A meaningful minority of negative feedback remains, indicating specific operational pain points rather than systemic failure.